# CDS 部署版：台风追踪与提取（历史业务预报）

本 Notebook 面向 CDS JupyterLab（4 CPU / 16GB），包含：依赖安装、数据请求、并行追踪、结果汇总。

稳定性策略：限并发、分批处理、失败重试、逐批落盘，避免卡死和内存崩溃。

In [ ]:
import os
import sys
import json
import time
import logging
import subprocess
from pathlib import Path
from datetime import datetime, timedelta

PROJECT_ROOT = Path('/home/jovyan/TianGong-AI-Cyclone') if Path('/home/jovyan/TianGong-AI-Cyclone').exists() else Path.cwd()
os.chdir(PROJECT_ROOT)

# 确保本地 src 包可导入（主进程 + 子进程继承）
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RAW_DIR = PROJECT_ROOT / 'data' / 'ecmwf_raw'
NC_DIR = PROJECT_ROOT / 'data' / 'ecmwf_nc'
TRACK_DIR = PROJECT_ROOT / 'track_output_cds'
ENV_DIR = PROJECT_ROOT / 'output' / 'forecast_environment'
SUMMARY_DIR = PROJECT_ROOT / 'output' / 'cds_notebook_summary'
STATE_DIR = PROJECT_ROOT / 'output' / 'cds_notebook_state'
for p in [RAW_DIR, NC_DIR, TRACK_DIR, ENV_DIR, SUMMARY_DIR, STATE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CPU_COUNT = os.cpu_count() or 4
PIPELINE_WORKERS = 2

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
print(f'PROJECT_ROOT={PROJECT_ROOT}')
print(f'SRC_DIR={SRC_DIR}')
print(f'CPU_COUNT={CPU_COUNT}, PIPELINE_WORKERS={PIPELINE_WORKERS}')

In [ ]:
def pip_install(packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', *packages]
    print('>>>', ' '.join(cmd))
    subprocess.check_call(cmd)

pip_install(['-r', 'requirements.txt'])
pip_install(['ecmwf-api-client', 'cfgrib', 'eccodes', 'tenacity', 'psutil'])
print('依赖安装完成（TIGGE WebAPI: ecmwf-api-client）')

## 一键双源流水线（AWS → TIGGE）

主运行单元会按顺序执行：
1. AWS Open Data 的 ECMWF 预报
2. TIGGE WebAPI 的 ECMWF 预报

每个源都执行闭环：请求数据 → 追踪 → 环境提取 → 删除 GRIB/NC 中间文件。


In [ ]:
import os
import sys
import subprocess
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path('/home/jovyan/TianGong-AI-Cyclone') if Path('/home/jovyan/TianGong-AI-Cyclone').exists() else Path.cwd()
os.chdir(PROJECT_ROOT)

SOURCES = 'aws,tigge'  # 固定顺序: 先 AWS 后 TIGGE
START_DATE = '2021-01-01'
END_DATE = datetime.utcnow().strftime('%Y-%m-%d')
RUN_HOURS = '0,12'
STEPS = ','.join(str(s) for s in range(0, 241, 6))
WORKERS = 2
MATCH_TOL_HOURS = 6
CLEAN_INTERMEDIATES_BEFORE_RUN = True
KEEP_FAILED_ARTIFACTS = False

cmd = [
    sys.executable,
    str(PROJECT_ROOT / 'scripts' / 'ec_dual_pipeline.py'),
    '--sources', SOURCES,
    '--start-date', START_DATE,
    '--end-date', END_DATE,
    '--run-hours', RUN_HOURS,
    '--steps', STEPS,
    '--workers', str(WORKERS),
    '--match-tol-hours', str(MATCH_TOL_HOURS),
    '--initials-csv', str(PROJECT_ROOT / 'input' / 'western_pacific_typhoons_superfast.csv'),
]

if CLEAN_INTERMEDIATES_BEFORE_RUN:
    cmd.append('--cleanup-intermediates-before-run')
else:
    cmd.append('--no-cleanup-intermediates-before-run')

if KEEP_FAILED_ARTIFACTS:
    cmd.append('--keep-failed-artifacts')

print('即将执行双源流水线命令:')
print(' '.join(cmd))

res = subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=False)
if res.returncode != 0:
    raise RuntimeError(f'双源流水线执行失败, exit={res.returncode}')

print('双源流水线执行完成。')


In [ ]:
# 状态检查：同时检查 AWS 与 TIGGE 两个状态文件
import json
from pathlib import Path

state_dir = PROJECT_ROOT / 'output' / 'cds_notebook_state'
state_files = [
    state_dir / 'pipeline_state_aws.json',
    state_dir / 'pipeline_state_tigge.json',
]

for sf in state_files:
    if not sf.exists():
        print(f'{sf.name}: 尚未生成')
        continue
    with open(sf, 'r', encoding='utf-8') as fr:
        st = json.load(fr)
    done_n = len(st.get('done', []))
    failed_n = len(st.get('failed', []))
    print(f'{sf.name}: done={done_n}, failed={failed_n}')
    if failed_n:
        print('  失败样例(前10):', st['failed'][:10])


In [ ]:
# 快速自检：验证子进程能导入 initial_tracker + ecmwfapi
from concurrent.futures import ProcessPoolExecutor
from multiprocessing import get_context

def _worker_env_check(project_root: str):
    import os
    import sys
    from pathlib import Path

    root = Path(project_root)
    os.chdir(root)
    src_dir = root / 'src'
    if str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))

    import initial_tracker
    import ecmwfapi
    has_webapi_cfg = (Path.home() / '.ecmwfapirc').exists()

    return {
        'initial_tracker': getattr(initial_tracker, '__file__', 'imported'),
        'ecmwfapi_available': hasattr(ecmwfapi, 'ECMWFDataServer'),
        'ecmwfapirc_exists': has_webapi_cfg,
    }

with ProcessPoolExecutor(max_workers=1, mp_context=get_context('fork')) as ex:
    result = ex.submit(_worker_env_check, str(PROJECT_ROOT)).result(timeout=120)
print('子进程环境检查通过:', result)

In [ ]:
# 旧流程已停用：请使用上方闭环并行流水线单元。
print('该单元已停用。请运行上方闭环流水线单元执行下载->转换->追踪->提取->删除。')

In [ ]:
import pandas as pd

track_files = sorted(TRACK_DIR.glob('*.csv'))
env_files = sorted(ENV_DIR.glob('*.json'))
if not track_files:
    raise RuntimeError('没有找到追踪输出，请先运行闭环主流程单元')

frames = []
for f in track_files:
    try:
        df = pd.read_csv(f)
        if not df.empty:
            df['track_file'] = f.name
            frames.append(df)
    except Exception as e:
        print(f'skip bad track file {f.name}: {e}')

all_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
summary = {
    'track_files': len(track_files),
    'environment_json_files': len(env_files),
    'loaded_files': len(frames),
    'total_points': int(len(all_df)),
    'time_min': str(all_df['time'].min()) if 'time' in all_df.columns and not all_df.empty else None,
    'time_max': str(all_df['time'].max()) if 'time' in all_df.columns and not all_df.empty else None,
    'min_msl': float(all_df['msl'].min()) if 'msl' in all_df.columns and not all_df.empty else None,
    'max_wind': float(all_df['wind'].max()) if 'wind' in all_df.columns and not all_df.empty else None,
    'pipeline_state': str(STATE_DIR / 'pipeline_state.json'),
}

summary_path = SUMMARY_DIR / 'tracking_summary.json'
with open(summary_path, 'w', encoding='utf-8') as fw:
    json.dump(summary, fw, ensure_ascii=False, indent=2)

display(pd.DataFrame([summary]))
print('摘要写入:', summary_path)

## 运行要点

- 数据源策略：优先使用 ECMWF TIGGE WebAPI（对应 `apps.ecmwf.int/datasets/data/tigge`）
- 目标时间范围默认 2021-01-01 到 2026-12-31（自动截断到当前日期）
- 仅对“有初始点匹配可能（±6h）”的 00Z/12Z 时次发起下载，避免无效任务
- 每个任务请求 0-240h、每 6h 一步（共 41 步）
- 下载优先 NetCDF；若服务端不支持则自动回退到 GRIB，再本地转换为 NC
- 追踪输出后，自动调用 `TCEnvironmentalSystemsExtractor` 生成环境场 JSON 到 `output/forecast_environment`
- 初始点 CSV 在任务内会自动补齐 `dt` 列（由 `datetime` 解析）
- 无匹配(0 tracks)或失败都可清理中间文件；默认失败也清理（`KEEP_FAILED_ARTIFACTS=False`）
- 并行固定 `PIPELINE_WORKERS=2`，使用 `fork` 模式（兼容 Jupyter），状态写入 `pipeline_state_tigge.json`
- 运行前请确保 `~/.ecmwfapirc` 可用；如需从头重跑，删除 `output/cds_notebook_state/pipeline_state_tigge.json`